In [33]:
import importlib.util as imp
import subprocess
import sys

import itk
import numpy as np

from Minder3D.sovImageProcessLogic import ImageProcessLogic
from Minder3D.sovUtils import time_and_log

import nibabel as nib
from totalsegmentator.python_api import totalsegmentator
from PySide6.QtWidgets import QApplication, QMainWindow, QVBoxLayout, QWidget, QPushButton, QLabel, QMessageBox, QComboBox
from PySide6.QtCore import Qt

class Logic:
    def __init__(self):
        self.ai_first_run = True
        self.image = None
        self.pre_image = None
        self.mask = None

    @time_and_log
    def initialize(self, image):
        """Initialize the AI model with the given image.

        This method initializes the AI model with the given image. It first checks for the presence of required dependencies and GPU support, and then sets the input image for further processing.

        Args:
            image: The input image for initializing the AI model.

        Returns:
            tuple: A tuple containing the status of initialization (bool), a message (str), and a flag to ask for user confirmation (bool).
        """

        if self.ai_first_run and imp.find_spec('totalsegmentator') is None:
            self.ai_first_run = False
            status = False
            msg = 'This method works best with NVidia GPUs.\nFirst install CUDA (https://developer.nvidia.com/cuda-downloads)\nand then install PyTorch (https://pytorch.org/get-started/locally/)'
            ask_to_continue = False
            return status, msg, ask_to_continue

        if imp.find_spec('torch') is None:
            status = False
            msg = 'PyTorch not found:\nFirst install CUDA (https://developer.nvidia.com/cuda-downloads)\nand then PyTorch (https://pytorch.org/get-started/locally/)'
            ask_to_continue = False
            return status, msg, ask_to_continue

        import torch

        if not torch.cuda.is_available():
            status = False
            msg = 'WARNING: PyTorch installed without CUDA support.\nThe AI methods will run on the CPU and be very slow.\nContinue?'
            ask_to_continue = True
            return status, msg, ask_to_continue

        self.image = image

        status = True
        msg = ''
        ask_to_continue = False
        return status, msg, ask_to_continue

    def preprocess(self):
        """Preprocesses the input image for further analysis.

        If the input image is None, returns None. If the 'totalsegmentator' module is not found, it installs it using pip.
        Then, it checks the spacing of the image and if it does not meet the specified conditions, it preprocesses the image
        using ImageProcessLogic.make_iso method with a spacing of 1.5. Otherwise, it uses the original image for preprocessing.

        Returns:
            SimpleITK.Image: The preprocessed image.

        Raises:
            subprocess.CalledProcessError: If the installation of 'TotalSegmentator' fails.
        """

        if self.image is None:
            return None

        if imp.find_spec('totalsegmentator') is None:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', 'TotalSegmentator']
            )

        spacing = self.image.GetSpacing()
        if not (
            spacing[0] == 1.5
            and spacing[0] == spacing[1]
            and spacing[1] == spacing[2]
        ):
            preproc = ImageProcessLogic()
            self.pre_image = preproc.make_iso(self.image, 1.5)
        else:
            self.pre_image = self.image
        return self.pre_image

    def run(self, body_part, target_body_part):
        """Perform image preprocessing and segmentation using the TotalSegmentator library.

        If the pre_image is not provided, it will be preprocessed before segmentation.
        The preprocessing step includes converting the input image to a NIfTI format and applying a transformation matrix.

        Returns:
            itk.Image: The segmented image after preprocessing and segmentation.
        """

        if self.pre_image is None:
            self.preprocess()
            if self.pre_image is None:
                return None


        pre_array = itk.GetArrayFromImage(self.pre_image)
        mat = np.eye(4)
        mat = mat * 1.5
        mat[3, 3] = 1
        nifti_nib = nib.Nifti1Image(
            np.transpose(pre_array, (2, 1, 0)).copy(), mat
        )
        seg_nib = totalsegmentator(
            input=nifti_nib, output=None, task=body_part)
      

        seg_array = seg_nib.get_fdata().astype(np.uint8)
        seg_image = itk.GetImageFromArray(
            np.transpose(seg_array, (2, 1, 0)).copy()
        )
        seg_image.CopyInformation(self.pre_image)
        
        return seg_image
        
# Converts image to nifti format, runs and returns the segmented image
class SegmentationPanelWidget(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.logic = Logic()
        self.setWindowTitle("Tool for Segmentation")
        self.main_layout = QVBoxLayout(self)
        self.selected_category = None
        self.init_ui()
        
    # Initialize button to select which category
    def init_ui(self):
        self.category_label = QLabel("Select Category to Segmentate:")
        self.main_layout.addWidget(self.category_label)
        self.category_buttons = {}
        
        # Creates a a dropdown menu with all of the categories available
        self.category_combobox = QComboBox()
        self.category_combobox.addItems(predefined_operations.keys())
        self.category_combobox.currentIndexChanged.connect(self.show_body_parts)
        self.main_layout.addWidget(self.category_combobox)

        # Allows the user to pick a body part from the avaible options does not become avaible until the category has been selected
        self.body_part_combobox = QComboBox()
        self.body_part_combobox.currentIndexChanged.connect(self.show_target_body_part_dropdown)
        self.body_part_combobox.setVisible(False)
        self.main_layout.addWidget(self.body_part_combobox)

        # Allows the user to pick the target body part (statistic they want to calculate) from the provided options. Does not become available until the category has been selected
        self.target_body_part_combobox = QComboBox()
        self.target_body_part_combobox.setVisible(False)
        self.main_layout.addWidget(self.target_body_part_combobox)

        # Initiates the segmentation process
        self.segment_button = QPushButton("Segmentate")
        self.segment_button.clicked.connect(self.segment)
        self.segment_button.setVisible(False)
        self.main_layout.addWidget(self.segment_button)


        self.status_label = QLabel("")
        self.status_label.setAlignment(Qt.AlignCenter)
        self.main_layout.addWidget(self.status_label)
    
    # Shows the body parts specific to the category selected
    def show_body_parts(self, category):
        slef.selected_category = category
        self.body_part_combobox.clear()
        self.body_part_combobox.addItems(body_parts[category])
        self.body_part_combobox.setVisible(True)
        self.target_body_part_combobox.setVisible(False)
        self.segment_button.setVisible(False)
        
    # Shows the avaible statistics to calculate based off of the previous selections
    def show_target_body_part_dropdown(self):
        self.target_body_part_combobox.clear()
        self.target_body_part_combobox.addItems(statistical_data)
        self.target_body_part_combobox.setVisible(True)
        self.segment_button.setVisible(True)
        
   # Runs the segmentation process
    def segment(self):
        category = self.category_combobox.currentText()
        body_part = self.body_part_combobox.currentText()
        target_body_part = self.target_body_part_combobox.currentText()

        # Update the status label with the selections that have been made
        self.status_label.setText(f"Category: {category}\nBody Part: {body_part}\nTarget Body Part: {target_body_part}")

        # Call Segmentation using the parameters selected 
        segmented_image = self.logic.run(body_part = body_part, target_body_part = target_body_part)
        if segmented_image:
            QMessageBox.information(self, "Segmentation Done", f"Segmentation of {body_part} with target_body_part {target_body_part} is complete.")
        else:
            QMessageBox.warning(self, "Segmentation Failed")

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Segmentation Tool")
        self.setCentralWidget(SegmentationPanelWidget(self))

if __name__ == "__main__":
    body_parts = {
        "Anatomical Structures":['adrenal_gland_left', 'adrenal_gland_right', 'aorta', 'brain', 'colon', 'duodenum', 'esophagus', 'face', 'femur_left', 'femur_right', 'gallbladder', 'heart', 'kidney_left', 'kidney_right', 'liver', 'lung_left', 'lung_right', 'pancreas', 'rib_left', 'rib_right', 'sacrum', 'scapula_left', 'scapula_right', 'small_bowel', 'spleen', 'stomach', 'trachea', 'urinary_bladder'],
        "Morphologically Altered Structures":['cyst', 'tumor', 'infiltrate', 'hemorrhage', 'necrosis'],
        "Physical Objects":['implant', 'foreign_body'],
        "Tissues":[ 'connective_tissue', 'muscle', 'nerve', 'fat']
    }

    statistical_data = ['Surface Area', 'Volume', 'Density']
        

    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())

ModuleNotFoundError: No module named 'Minder3D'